## Example Use case of Custom Model and the original LongT5 Model  

In [1]:
import torch
from base_model import KATSum
from transformers import LongT5ForConditionalGeneration,AutoTokenizer
from kg_embedder import KGEncoder
from datasets import load_dataset,load_from_disk
from torch.utils.data import Dataset, DataLoader

/home/shreesh/Documents/project_repositories/nlp-kg-summarization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "google/long-t5-tglobal-base"
NUM_SIDECAR_LAYERS = 4 # [1..12]
DEVICE = "cpu"

### Ensure dataset folder is created along with appropriate name of folder

### These datasets can be concatenated to a single dataset if needed as no training is done

In [3]:
train_dataset = load_from_disk("./dataset/pubmed_with_triples_v/train")
val_dataset = load_from_disk("./dataset/pubmed_with_triples_v/validation")
test_dataset = load_from_disk("./dataset/pubmed_with_triples_v/test")

print(f"Number of train samples of {len(train_dataset)}")
print(f"Number of validation samples of {len(val_dataset)}")
print(f"Number of test samples of {len(test_dataset)}")

Number of train samples of 19525
Number of validation samples of 2499
Number of test samples of 2500


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = LongT5ForConditionalGeneration.from_pretrained(MODEL_NAME)

kg_embedder = KGEncoder(
    encoder=base_model.encoder,
    tokenizer=tokenizer,
    hidden_dim=base_model.config.d_model,
    device=DEVICE,
)

custom_model = KATSum(
    base_model=base_model,
    kg_embedder=kg_embedder,
    num_sidecar_layers=NUM_SIDECAR_LAYERS,
    fusion_gate_biases=None, # Can be customized (See KGSidecarLayer)
    device=DEVICE,
)

custom_model.parameter_count()

/home/shreesh/Documents/project_repositories/nlp-kg-summarization/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Number of decoder layers for KATSUM model: 12
Sidecar indices: [8, 9, 10, 11]
Number of sidecar layers: 4
Total parameters: 247,590,532
Trainable parameters: 3,076 (0.0%)
Frozen parameters: 247,587,456 (100.0%)
KG Embedder params: 0
Sidecar layer params: 28,320,772
Num sidecar layers: 4


### Custom LongT5 with KG embeddings

In [ ]:
generation_config = {
    "max_new_tokens": 300,  # Give it room to finish the thought
    "num_beams": 4,  # Standard sweet spot for quality vs speed
    "min_length": 40,  # Prevent "one-sentence" lazy summaries
    "length_penalty": 2.0,  # Higher (>1.0) encourages longer, complete sentences
    "no_repeat_ngram_size": 3,  # Prevents the "participants... participants" loop
    "early_stopping": True,  # Stop as soon as all beams hit the </s> token
    "repetition_penalty": 2.5,  # Heavily discourages copying the same phrases
}

tokens = tokenizer(val_dataset[0]["article"], return_tensors="pt", padding=True)

output_ids = custom_model.generate_summary(
    input_ids=tokens["input_ids"],
    attention_mask=tokens["attention_mask"],
    triples=val_dataset[0]["rebel_triples"],
    **generation_config
)

tokenizer.decode(token_ids=output_ids[0], skip_special_tokens=True)

/home/shreesh/Documents/project_repositories/nlp-kg-summarization/.venv/lib/python3.11/site-packages/transformers/modeling_utils.py:949: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


### LongT5 Base Model

In [ ]:
output_ids = base_model.generate(
    input_ids=tokens["input_ids"],
    attention_mask=tokens["attention_mask"],
    **generation_config
)

tokenizer.decode(token_ids=output_ids[0], skip_special_tokens=True)